# Train YOLO11 on VisDrone Vehicles

Before running, choose `Runtime > Change runtime type > GPU` in Colab.

This notebook downloads VisDrone DET train/val, converts the annotations to YOLO format, and trains YOLO11 on three size-focused classes: `car`, `big_car`, and `motorcycle`. VisDrone van/truck/bus are merged into `big_car`; smaller two/three-wheel vehicle classes are merged into `motorcycle`.

The final `best.pt` is saved in Google Drive under `MyDrive/visdrone_yolo_runs/visdrone_vehicles_yolo11s/weights/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics opencv-python

In [ ]:
from pathlib import Path

RAW_ROOT = Path('/content/datasets/visdrone_raw')
DATA_ROOT = Path('/content/datasets/visdrone_vehicles')
DATA_YAML = Path('/content/datasets/visdrone_vehicles.yaml')
RUN_ROOT = Path('/content/drive/MyDrive/visdrone_yolo_runs')

EPOCHS = 100
IMG_SIZE = 1280
BATCH = 4
MODEL = 'yolo11s.pt'
RUN_NAME = 'visdrone_vehicles_yolo11s'

RAW_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess

urls = {
    'VisDrone2019-DET-train.zip': 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip',
    'VisDrone2019-DET-val.zip': 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',
}

for name, url in urls.items():
    zip_path = RAW_ROOT / name
    if not zip_path.exists():
        subprocess.run(['wget', '-q', '--show-progress', '-O', str(zip_path), url], check=True)
    subprocess.run(['unzip', '-q', '-o', str(zip_path), '-d', str(RAW_ROOT)], check=True)

print('Raw VisDrone folders:')
for path in sorted(RAW_ROOT.glob('VisDrone2019-DET-*')):
    print(' ', path)

In [ ]:
import shutil
from collections import Counter

import cv2

RAW_ID_TO_CLASS_NAME = {
    3: 'motorcycle',
    4: 'car',
    5: 'big_car',
    6: 'big_car',
    7: 'motorcycle',
    8: 'motorcycle',
    9: 'big_car',
    10: 'motorcycle',
}
CLASS_NAMES = ['car', 'big_car', 'motorcycle']
CLASS_NAME_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IMAGE_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']

def reset_dir(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def clamp(value):
    return min(max(value, 0.0), 1.0)

def image_size(path):
    image = cv2.imread(str(path))
    if image is None:
        raise RuntimeError(f'Could not read image: {path}')
    height, width = image.shape[:2]
    return width, height

def find_image(images_dir, stem):
    for ext in IMAGE_EXTENSIONS:
        candidate = images_dir / f'{stem}{ext}'
        if candidate.exists():
            return candidate
    raise RuntimeError(f'Missing image for annotation: {stem}')

def convert_split(split, raw_dir, out_root):
    images_dir = raw_dir / 'images'
    annotations_dir = raw_dir / 'annotations'
    out_images = out_root / 'images' / split
    out_labels = out_root / 'labels' / split
    reset_dir(out_images)
    reset_dir(out_labels)

    class_counts = Counter()
    image_count = 0
    box_count = 0

    for annotation_path in sorted(annotations_dir.glob('*.txt')):
        src_image = find_image(images_dir, annotation_path.stem)
        width, height = image_size(src_image)
        label_lines = []

        for raw_line in annotation_path.read_text(encoding='utf-8').splitlines():
            parts = [part.strip() for part in raw_line.split(',')]
            if len(parts) != 8:
                continue
            bbox_left, bbox_top, bbox_width, bbox_height, score, category, truncation, occlusion = parts
            if int(score) == 0:
                continue
            class_name = RAW_ID_TO_CLASS_NAME.get(int(category))
            if class_name is None:
                continue

            x = float(bbox_left)
            y = float(bbox_top)
            w = float(bbox_width)
            h = float(bbox_height)
            if w <= 0 or h <= 0:
                continue

            x_center = clamp((x + w / 2.0) / width)
            y_center = clamp((y + h / 2.0) / height)
            norm_w = clamp(w / width)
            norm_h = clamp(h / height)
            if norm_w == 0.0 or norm_h == 0.0:
                continue

            class_index = CLASS_NAME_TO_INDEX[class_name]
            label_lines.append(f'{class_index} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}')
            class_counts[class_name] += 1
            box_count += 1

        shutil.copy2(src_image, out_images / src_image.name)
        (out_labels / f'{annotation_path.stem}.txt').write_text('\n'.join(label_lines) + ('\n' if label_lines else ''), encoding='utf-8')
        image_count += 1

    print(f'{split}: {image_count} images, {box_count} vehicle boxes')
    print(dict(class_counts))
    return class_counts

train_counts = convert_split('train', RAW_ROOT / 'VisDrone2019-DET-train', DATA_ROOT)
val_counts = convert_split('val', RAW_ROOT / 'VisDrone2019-DET-val', DATA_ROOT)

DATA_YAML.write_text(
    'path: /content/datasets/visdrone_vehicles\n'
    'train: images/train\n'
    'val: images/val\n\n'
    'names:\n'
    '  0: car\n'
    '  1: big_car\n'
    '  2: motorcycle\n',
    encoding='utf-8',
)
print(DATA_YAML.read_text())

In [ ]:
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Switch Colab to Runtime > Change runtime type > GPU before training.'
print(torch.cuda.get_device_name(0))

model = YOLO(MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    pretrained=True,
    project=str(RUN_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    patience=20,
)

best = RUN_ROOT / RUN_NAME / 'weights' / 'best.pt'
print(f'Best weights: {best}')

In [ ]:
best = RUN_ROOT / RUN_NAME / 'weights' / 'best.pt'
model = YOLO(str(best))
model.val(data=str(DATA_YAML), imgsz=IMG_SIZE, batch=BATCH, device=0)
print(f'Use this best.pt in your local project: {best}')

In [ ]:
import shutil

simple_best = Path('/content/drive/MyDrive/visdrone_vehicle_best.pt')
shutil.copy2(best, simple_best)
print(f'Convenience copy: {simple_best}')